# A3-M3-2 · RAG평가와 리랭킹
교안의 같은 번호 단원과 함께 봅니다. 노트북은 01 → 02 → 03 순서로 실행합니다.

## 04-1. 준비 — 컬렉션 열고 search · evaluate 만들기

In [ ]:
import time
import numpy as np
import pandas as pd
import chromadb
from emb import embed, RERANK_MODEL

client = chromadb.PersistentClient(path="chroma_db")
col = client.get_collection("shop_faq")
tests = pd.read_csv("shop_faq_tests.csv")

def search(question, k=3, where=None):
    r = col.query(query_embeddings=embed([question], kind="query"), n_results=k, where=where)
    return pd.DataFrame({
        "faq_id": [int(i) for i in r["ids"][0]],
        "score": [round(1 - d, 3) for d in r["distances"][0]],
        "category": [m["category"] for m in r["metadatas"][0]],
        "question": [m["question"] for m in r["metadatas"][0]],
        "doc": r["documents"][0],
    })

def evaluate(fn, **kw):
    rows = []
    for q, gold in tests.itertuples(index=False):
        ranked = fn(q, **kw)["faq_id"].tolist()
        rows.append({"question": q, "gold": gold, "top1": ranked[0],
                     "rank": ranked.index(gold) + 1 if gold in ranked else np.nan})
    return pd.DataFrame(rows)

def summary(ev):
    return {"Hit@1": round(float((ev["rank"] == 1).mean()), 3), "Hit@3": round(float((ev["rank"] <= 3).mean()), 3),
            "MRR": round(float((1 / ev["rank"]).fillna(0).mean()), 3)}

print("FAQ", col.count(), "건 | 평가 질문", len(tests), "개")

## 04-2. 평가 질문 12개로 Hit@1 · Hit@3 · MRR

In [ ]:
ev = evaluate(search, k=5)
print(ev.assign(rank=ev["rank"].astype("Int64")).to_string(index=False))
print(summary(ev))

## 04-3. 틀린 질문 보기

In [ ]:
for r in ev[ev["rank"] != 1].itertuples(index=False):
    rank = "5위 밖" if pd.isna(r.rank) else f"{int(r.rank)}위"
    print(f"\nQ: {r.question} | 정답 {r.gold} | 1위 {r.top1} | 정답 {rank}")
    print(search(r.question, k=3)[["faq_id", "score", "category", "question"]].to_string(index=False))

## 05-1. 리랭커 붙이기

In [ ]:
from transformers import pipeline
rr = pipeline("text-classification", model=RERANK_MODEL, device=-1)

def search_rerank(question, k=3, pool=8, where=None):
    cand = search(question, k=pool, where=where)
    pairs = [{"text": question, "text_pair": d} for d in cand["doc"]]
    cand["rerank"] = [round(s["score"], 3) for s in rr(pairs, function_to_apply="none")]
    return cand.sort_values("rerank", ascending=False).head(k)

q = "비번 까먹었어요"
print("[Chroma 순서]")
print(search(q, k=5)[["faq_id", "score", "question"]].to_string(index=False))
print("\n[리랭킹 순서]")
print(search_rerank(q, k=5, pool=5)[["faq_id", "score", "rerank", "question"]].to_string(index=False))

## 05-2. 검색만 vs 검색 + 리랭킹 비교

In [ ]:
ev_rr = evaluate(search_rerank, k=5, pool=8)
print(pd.DataFrame({"검색만": summary(ev), "검색+리랭킹": summary(ev_rr)}).to_string())

for name, fn in [("검색만", lambda q: search(q, 5)), ("검색+리랭킹", lambda q: search_rerank(q, 5, 8))]:
    t = time.perf_counter()
    for q in tests["question"]:
        fn(q)
    print(f"{name:<8} 질문당 {(time.perf_counter() - t) / len(tests) * 1000:.0f} ms")